# Inference Layer - Complete Feature Showcase

This notebook demonstrates all features of the low-level inference layer.

**Key components:**
- `create_client()`: Factory function for UnifiedInferenceClient
- `run_completion()`: Execute single inference request
- `run_batch()`: Batch processing with checkpoint/resume
- `InferenceRequest/Result`: Type-safe request/response objects

## 1. Setup & Imports

In [ ]:
from inference import (
    create_client,
    run_completion,
    run_batch,
    InferenceRequest,
    InferenceResult,
    InferenceConfig,
)

from pathlib import Path

## 2. Configuration

Configuration is loaded from YAML. See `config/inference.example.yaml` for structure.

In [ ]:
config_path = Path("config/inference.example.yaml")

if config_path.exists():
    print("Configuration file:")
    print("=" * 50)
    with open(config_path) as f:
        for i, line in enumerate(f):
            if i >= 50:
                print("...")
                break
            print(line.rstrip())
else:
    print(f"Config not found at {config_path}")

## 3. Single Completion

In [ ]:
async def single_completion_demo():
    client = create_client("config/inference.example.yaml")
    print(f"✓ Client created: {type(client).__name__}")
    
    request = InferenceRequest(
        model_alias="mock-test",
        prompt="What is the capital of France?",
        max_tokens=100,
        temperature=0.7,
    )
    print(f"✓ Request: model_alias={request.model_alias}")
    
    result = await run_completion(client, request)
    
    print("\n" + "=" * 50)
    print("RESULT")
    print("=" * 50)
    print(f"Content: {result.content}")
    print(f"Provider: {result.provider}")
    print(f"Model: {result.model}")
    print(f"Tokens: {result.prompt_tokens} prompt + {result.completion_tokens} completion")
    print(f"Latency: {result.latency_ms:.2f} ms")
    print(f"Retries: {result.retry_count}")
    return result

result = await single_completion_demo()

## 4. Batch Processing

**Features:** checkpointing, resume on interruption, structured logging.

In [ ]:
async def generate_batch_requests():
    prompts = [
        "What is 2+2?",
        "What is the capital of Germany?",
        "Name a primary color.",
    ]
    for i, prompt in enumerate(prompts):
        print(f"  Yielding request {i+1}/{len(prompts)}")
        yield InferenceRequest(model_alias="mock-test", prompt=prompt, max_tokens=50)

async def batch_demo():
    print("Starting batch...")
    await run_batch("config/inference.example.yaml", generate_batch_requests())
    print("✓ Batch completed!")
    print("  Logs: logs/inference.jsonl")
    print("  Checkpoints: checkpoints/batch.jsonl")

await batch_demo()

## 5. Error Handling

Invalid model aliases raise exceptions. Handle with try/except.

In [ ]:
async def error_demo():
    client = create_client("config/inference.example.yaml")
    
    # Invalid model alias
    print("Example: Invalid Model Alias")
    print("-" * 40)
    try:
        request = InferenceRequest(model_alias="nonexistent-model", prompt="Hello")
        result = await run_completion(client, request)
    except Exception as e:
        print(f"✓ Caught {type(e).__name__}: {e}")
    
    print()
    
    # Valid request
    print("Example: Valid Request")
    print("-" * 40)
    request = InferenceRequest(model_alias="mock-test", prompt="Hello")
    result = await run_completion(client, request)
    print(f"✓ Success: {result.content[:50]}...")

await error_demo()

## 6. Log Analysis

Logs contain metadata (provider, model, tokens, latency, errors) - NOT prompt/response content.

In [ ]:
import json

def analyze_logs():
    log_path = Path("logs/inference.jsonl")
    
    if not log_path.exists():
        print(f"No log file at {log_path}")
        return
    
    entries = [json.loads(line) for line in open(log_path) if line.strip()]
    print(f"Log entries: {len(entries)}")
    
    success = sum(1 for e in entries if e.get("status") == "success")
    failures = sum(1 for e in entries if e.get("status") == "failure")
    print(f"  Success: {success}, Failures: {failures}")
    
    if entries:
        print("\nSample entry:")
        for entry in reversed(entries):
            if entry.get("status") == "success":
                print(json.dumps(entry, indent=2))
                break
    
    # Checkpoint analysis
    checkpoint_path = Path("checkpoints/batch.jsonl")
    if checkpoint_path.exists():
        checkpoints = [json.loads(line) for line in open(checkpoint_path) if line.strip()]
        completed = sum(1 for c in checkpoints if c.get("status") == "success")
        print(f"\nCheckpoints: {len(checkpoints)} entries, {completed} successful")

analyze_logs()

## Summary

| Feature | Function | Purpose |
|---------|----------|---------|
| Client | `create_client()` | Initialize from config |
| Single | `run_completion()` | Execute one request |
| Batch | `run_batch()` | Process multiple with checkpointing |
| Request | `InferenceRequest` | model_alias, prompt, options |
| Result | `InferenceResult` | content, tokens, latency |

**Next:** `experiments_example.ipynb` for high-level experiment workflows.